# Train a Cellpose model

**What it does.** Fine-tune a Cellpose segmentation model on your own annotated images.

**When to use it.** When the stock Cellpose models mis-segment your cells — unusual morphology, unusual magnification, or a stain they were not trained on.

**What you get.** A model file you can point the masking step at.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.submodules.train_cellpose`

```
train_cellpose(settings)
```

Fine-tune the Cellpose-SAM (``cpsam``) segmentation model from images and paired masks.

In [ ]:
from spacr.submodules import train_cellpose

## 3. Settings

`spacr.settings.get_train_cellpose_default_settings` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import get_train_cellpose_default_settings

defaults = get_train_cellpose_default_settings({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

Every key, with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version by `tools/build_notebook_settings.py`, so these are the real keys, the real defaults and the real descriptions. Re-run that tool after upgrading spaCR.

In [ ]:
# Generated by tools/build_notebook_settings.py — do not edit by hand.
# Values are this installed version's defaults; the text above each key is
# its live description. Edit values in place -- a key left at its default
# behaves exactly as if it were absent.

# spacr.submodules.train_cellpose  (16 settings)
settings = {
    # (int) - Multiplier on background setting the signal threshold
    # (background * Signal_to_noise) used when normalising for Cellpose.
    # Per channel, spaCR takes the first of the 98th, 99th, 99.9th,
    # 99.99th and 99.999th percentiles that exceeds the threshold and
    # rescales to it. RAISE for less clipping and dimmer output; LOWER to
    # stretch faint objects harder at the cost of saturating bright ones.
    # If no percentile clears the threshold the range collapses to the 2nd
    # percentile, which is the sign the value is far too high. Ignored
    # when percentiles is set. Default 10 (5 in check_cellpose_models).
    'Signal_to_noise': 10,

    # (bool) - Expand the training split 8-fold by adding all four
    # 90-degree rotations of each crop plus their horizontal mirrors; the
    # validation and test splits are never augmented. Turn it on when you
    # have few annotated objects and validation accuracy lags training
    # accuracy. The expanded set is materialised in RAM, so expect roughly
    # 8x the memory and 8x the epoch time. Default False.
    'augment': False,

    # (float) - Per-channel background level in raw intensity units.
    # Pixels below it are zeroed when remove_background is on, and it is
    # multiplied by Signal_to_noise to set the upper anchor for
    # normalization. Raise it if faint haze survives; set it too high and
    # dim real objects vanish. Default 100 (200 for Cellpose training and
    # plaque analysis).
    'background': 200,

    # (int) - How many images are held and processed together in one pass:
    # field stacks during normalization and Cellpose segmentation, crops
    # per step during classifier training and activation maps. Raising it
    # speeds runs up but increases RAM/VRAM roughly linearly; lower it on
    # out-of-memory errors. Defaults: 50 for mask generation, 64 for
    # training.
    'batch_size': 8,

    # (float) - DEPRECATED. Expected object diameter in pixels, passed to
    # model.eval(diameter=...) by the mask-finetune tool and
    # check_cellpose_models. Cellpose resizes each image by 30/diameter so
    # objects match the network's ~30 px working size, so a value BELOW
    # the true size upscales the image and one above downscales it. Prefer
    # the per-object diameter settings; this one remains only for those
    # two tools. Default 30.
    'diameter': 30,

    # (bool) - Start from randomly initialised weights instead of fine-
    # tuning the pretrained model. Almost always leave OFF: fine-tuning
    # needs tens of images where from-scratch needs thousands. Default
    # False.
    'from_scratch': False,

    # (float) - Step size passed to the optimizer. Too high and the loss
    # spikes or flatlines at chance; too low and training crawls or
    # settles in a poor minimum. 1e-3 suits training from scratch, while
    # 1e-4 to 1e-5 is safer when fine-tuning ImageNet weights
    # (init_weights=True). The chosen schedule decays this starting value
    # over the run. Default 0.001.
    'learning_rate': 0.2,

    # (str) - Cellpose model to segment with. Cellpose 4 ships exactly
    # one, 'cpsam'; the pre-SAM names ('cyto', 'cyto2', 'cyto3', 'nuclei')
    # are accepted so old settings files load, but they are mapped to
    # 'cpsam' and reported, because Cellpose resolves them to cpsam
    # silently anyway. Of the three parameters that used to distinguish
    # models, only diameter still does anything under Cellpose 4 (eval
    # rescales the image by 30/diameter); model_type and diam_mean are
    # logged as 'not used in v4.0.1+' and dropped. Leave at 'cpsam' unless
    # you are loading a custom CPSAM checkpoint. Default 'cpsam'.
    'model_name': 'new_model',

    # (str) - Backbone architecture for the single-object image
    # classifier: any TorchVision classification model name (resnet50,
    # maxvit_t, densenet121, ...). An unrecognised name is NOT fatal when
    # it is read -- choose_model prints 'Invalid model_type' and returns
    # None, and training fails afterwards -- and the special name 'custom'
    # passes the name check then raises NotImplementedError. Bigger
    # backbones need more memory and more labelled crops to beat a smaller
    # one. Default 'maxvit_t'.
    'model_type': 'cpsam',

    # (int) - Number of training passes train_seg makes over the annotated
    # image/mask batch. It also sets the checkpoint interval (a model is
    # saved every n_epochs/10) and is written into the saved model
    # filename. Raise it for a better fit on large annotation sets; lower
    # it when a small set starts overfitting. Default 10000.
    'n_epochs': 10000,

    # (bool) - Hard-clip every pixel below the 'background' value to zero
    # before normalization and segmentation. Use it when a channel carries
    # a bright, even haze that inflates the normalization floor; leave it
    # off for dim or already flat-fielded data, since the clip silently
    # deletes faint real signal. Default False.
    'remove_background': False,

    # (bool or float) - Resize every image to target_height x target_width
    # before running Cellpose, then scale the returned mask back to the
    # original dimensions with nearest-neighbour interpolation so
    # measurements stay in original pixels. Turn it on to bring oversized
    # fields to the scale a model was trained at, or to cut GPU memory.
    # Requires target_height and target_width. Default False (True for
    # plaque analysis).
    'resize': False,

    # (int) - Edge length in pixels that the training images and masks are
    # resized to before Cellpose fine-tuning, applied to both axes so the
    # input becomes square. Larger keeps fine boundary detail and costs
    # VRAM and time roughly quadratically; smaller trains faster and blurs
    # exactly the outlines the model is being taught. Default 1000.
    'target_size': 1000,

    # (bool) - Print extra run detail instead of the minimal log: the
    # resolved settings table, the channel and model choices per object
    # type, per-table row counts, and how many objects survive each
    # filter. It only adds console output, so turn it on when object
    # counts come out unexpected and you need to see which stage removed
    # them. The default differs per pipeline -- True for mask, UMAP,
    # screen analysis, barcode mapping and Cellpose training; False for
    # measure, the plotting helpers and regression.
    'verbose': True,

    # (float) - L2 penalty applied to the weights on every optimizer step
    # (AdamW applies it decoupled from the gradient). Raise it, toward
    # 1e-3 to 1e-2, when validation loss climbs while training loss keeps
    # falling; lower it toward 0 when the model cannot fit the training
    # set at all. Every supported optimizer honours it. Default 0.00001.
    'weight_decay': 1e-05,

    # (list of int) - [width, height] in pixels that every crop is resized
    # to before it reaches the model, so a batch is uniform. Must match
    # what the model was trained on. Default [224, 224].
    'width_height': [1000, 1000],
}

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
train_cellpose(settings)

## Where the output went

A model file you can point the masking step at.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.